# AWS Bedrock AgentCore - Managed Session Storage via a Filesystem Configuration

This notebook demonstrates how to:
1. Create and deploy a Bedrock AgentCore agent w/ filesystem configuration
2. Invoke the agent with standard prompts to generate session related info into filesystem
3. Execute system commands directly in the agent runtime using [`invoke_agent_runtime_command`](https://docs.aws.amazon.com/boto3/latest/reference/services/bedrock-agentcore/client/invoke_agent_runtime_command.html) to check those info from filesystem
4. Execute command again from other session to show session isolation.

### Tutorial Details

| Information         | Details                                                   |
|:--------------------|:----------------------------------------------------------|
| Tutorial type       | Managed Session storage on Runtime                        |
| Tool type           | HTTP server                                               |
| Tutorial components | Hosting on AgentCore Runtime                              |
| Tutorial vertical   | Cross-vertical                                            |
| Example complexity  | Medium                                                    |
| SDK used            | Amazon BedrockAgentCore Python SDK and boto3              |


### Tutorial Architecture
In this tutorial notebook, you are going to build one agent. First you will deploy the agent on AgentCore Runtime with claude skill from `persistent-notes` folder. Then you will invoke it to generate local notes and check local files. Finally you will see session isolation in action.

So let's get started!

## Step 1: Install Dependencies

First, we install the required Python packages including boto3 and the Bedrock AgentCore SDK.

In [1]:
!uv pip install -qU -r requirements.txt

In [2]:
!uv pip freeze | grep boto3

Using Python 3.14.3 environment at: /Users/eric.fu/projects/xealth/agentcore-samples/.venv
boto3==1.42.84


**Restart your kernel before continue.**

## Step 2: Create IAM Execution Role

Create an IAM role with the necessary permissions for AgentCore Runtime:
- ECR access to pull container images
- CloudWatch Logs for logging
- Bedrock model invocation

In [4]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)
user_name = os.getenv("USER_NAME")
if not user_name:
    raise ValueError("USER_NAME environment variable is not set. Please set it in the .env file.")


In [5]:
# Create IAM ROLE
from helpers.utils import create_agentcore_runtime_execution_role, SAMPLE_ROLE_NAME

execution_role_arn = create_agentcore_runtime_execution_role(SAMPLE_ROLE_NAME)
execution_role_arn

ℹ️ Role SessionDemoBedrockAgentCoreRole already exists
Role ARN: arn:aws:iam::372080370602:role/SessionDemoBedrockAgentCoreRole


'arn:aws:iam::372080370602:role/SessionDemoBedrockAgentCoreRole'

---

## Step 3: Deployment

We're using Docker deployment for better dependency management and consistency.

### Make deployment (Build Docker and Push to ECR)

Now, let's build a docker image and push it to ECR, but firstly, let's check if repo exists, otherwise create it.

#### Setup AWS Clients

Initialize boto3 clients and get AWS account information needed for ECR operations.

In [6]:
import json
import boto3
import subprocess
import base64
from boto3.session import Session

boto_session = Session()
sts = boto3.client('sts')

account_id = sts.get_caller_identity()['Account']
region = boto_session.region_name
region

'us-west-2'

In [7]:
repo_name = 'managed-session-agent-demo-' + user_name
ecr = boto3.client('ecr', region_name=region)

try:
    response = ecr.create_repository(repositoryName=repo_name)
    repo_uri = response['repository']['repositoryUri']
    repo_arn = response['repository']['repositoryArn']
    print(f"✅ Created repository: {repo_uri}")
except ecr.exceptions.RepositoryAlreadyExistsException:
    print("ℹ️ Repository already exists")
    response = ecr.describe_repositories(repositoryNames=[repo_name])
    repo_uri = response['repositories'][0]['repositoryUri']
    repo_arn = response['repositories'][0]['repositoryArn']

print(f"Repository URI: {repo_uri}")
print(f"Repository ARN: {repo_arn}")

ℹ️ Repository already exists
Repository URI: 372080370602.dkr.ecr.us-west-2.amazonaws.com/managed-session-agent-demo-eric_fu
Repository ARN: arn:aws:ecr:us-west-2:372080370602:repository/managed-session-agent-demo-eric_fu


#### ECR Authentication

Get authorization token from ECR and login to Docker. This allows us to push images to our private ECR repository.

In [8]:
auth = ecr.get_authorization_token()
token = base64.b64decode(auth['authorizationData'][0]['authorizationToken']).decode().split(':')[1]
subprocess.run(f'echo {token} | docker login --username AWS --password-stdin {account_id}.dkr.ecr.{region}.amazonaws.com', shell=True)

Login Succeeded


CompletedProcess(args='echo eyJwYXlsb2FkIjoia2RkVDVJM1RyYXZBaEU0aEFzd1ZYOXFrOFhzcmd3Vjk4L2VkVHhIYnNPODlGaEsyK1lYaW1Ic0VTVkhZVkFzMTFVRjdUd2kzVVl5ZTVCbGZSSEJvY01QNCt1TUxldXpyZ2RWSnVpVFJ1TU1aWkcwQUtDN1NHRDA3ZExOS3lhTnhlSTlLODExZjdkWG4xeFA1d25hZ3MrVTh4aUx4ZXlrRm41YjZWZlJTRzdpNldsc3cxdklzTUJNbWpSVXdpNzY4VklTM0RuY3UrRE5rZm0rcE5ORmx3d2dicFE2QU9UdS80QldrS3dRby9iNnJtTDFEYThBWm8xRkg5Z3FYemZSRldRaTMzazl5U0U4YTRMbkcwUVE2Z2FjTGJsWU9SRm5YNHR0YlJjYlhtMGJXM05aeVJuZEVCM3dBOGg4REVBSExDL014WnJ6RHVNSXFaSE1ub0J4VG92eUhxSi9pRjduZnFXVWgxWDN1ZWpiSzllamgzZCthMzltdFdxSWpBeWpkTndTTi9TNVhneThGamprdUFlSEpjOVpjNVlkdnRJK1VjSERnSHo4dTVwblJveDlhVGxqZzFDandoYjluQm1FT1JtTmF2WWUvSW1HR2FBZ2tKWkovdVhUTjVhbXdUQ1BnS3BYT1c4eWQrV0RkZ1ZoMDlkSXJsQWJQVVFPU01hMC9CN1pvRmNCaXhYVWdYcWszaTFOSTFtcGdHdzJ2aDNxdFBaU1B2cjNhY1RkUWZ3RG1peElGQnJia2dTQXRvaHltclBTMEpYZ2ZnNE9VYld5V0ZSanF5Ym44cnVUK3RtV2tZTk95TzJCd1FKdmRZbGV2NXUzQ1ZhTlFkWlZvQjNnU0R4WU96MHhnZmo2VFlKeFREVmQ0RjBGWlhqcFJ6eDdocEtQaVBEdlN5L093RG11Mzh3QnJzeEhLSDVBa1FraFZCYnRtb0VuZ3dqcmcr

#### Build Docker Image

Build the Docker image locally using the Dockerfile in the current directory. This packages our agent code and dependencies.

In [17]:
docker_build = subprocess.run(['docker', 'build', '-t', f'{repo_name}:latest', '.'])

#0 building with "default" instance using docker driver

#1 [internal] load build definition from Dockerfile
#1 transferring dockerfile: 913B done
#1 DONE 0.0s

#2 [internal] load metadata for ghcr.io/astral-sh/uv:python3.14-bookworm-slim
#2 DONE 0.5s

#3 [internal] load .dockerignore
#3 transferring context: 2B done
#3 DONE 0.0s

#4 [internal] load build context
#4 transferring context: 1.82kB done
#4 DONE 0.0s

#5 [1/8] FROM ghcr.io/astral-sh/uv:python3.14-bookworm-slim@sha256:7cf77f594be8042dab6daa9fe326f90962252268b4f120a7f5dccce4d947e6c1
#5 resolve ghcr.io/astral-sh/uv:python3.14-bookworm-slim@sha256:7cf77f594be8042dab6daa9fe326f90962252268b4f120a7f5dccce4d947e6c1 0.0s done
#5 DONE 0.0s

#6 [4/8] RUN uv pip install -r requirements.txt
#6 CACHED

#7 [3/8] COPY requirements.txt requirements.txt
#7 CACHED

#8 [6/8] RUN useradd -m -u 1000 bedrock_agentcore
#8 CACHED

#9 [2/8] WORKDIR /app
#9 CACHED

#10 [5/8] RUN uv pip install aws-opentelemetry-distro
#10 CACHED

#11 [7/8] COPY ./per

#### Tag Docker Image

Tag the local image with the ECR repository URI so it can be pushed to ECR.

In [18]:
subprocess.run(['docker', 'tag', f'{repo_name}:latest', f'{repo_uri}:latest'])

CompletedProcess(args=['docker', 'tag', 'managed-session-agent-demo-eric_fu:latest', '372080370602.dkr.ecr.us-west-2.amazonaws.com/managed-session-agent-demo-eric_fu:latest'], returncode=0)

#### Push to ECR

Push the tagged image to ECR. This makes it available for AgentCore Runtime to pull and deploy.

Docker push

In [19]:
subprocess.run(['docker', 'push', f'{repo_uri}:latest'])

print(f"✅ Pushed to: {repo_uri}:latest")

The push refers to repository [372080370602.dkr.ecr.us-west-2.amazonaws.com/managed-session-agent-demo-eric_fu]
d3d5d8ab26d2: Waiting
52d45391171a: Waiting
bbd37776e49d: Waiting
3047ef430b99: Waiting
c7033875c9b1: Waiting
3f1031b52e55: Waiting
d8082048b068: Waiting
0bac6db50ef3: Waiting
8c9d24ec5c6e: Waiting
0c1d11d59da0: Waiting
85b498686cd0: Waiting
3c57f54fe1a6: Waiting
b378c443565f: Waiting
b378c443565f: Waiting
d3d5d8ab26d2: Waiting
52d45391171a: Waiting
bbd37776e49d: Waiting
3047ef430b99: Waiting
c7033875c9b1: Waiting
3f1031b52e55: Waiting
d8082048b068: Waiting
0bac6db50ef3: Waiting
8c9d24ec5c6e: Waiting
0c1d11d59da0: Waiting
85b498686cd0: Waiting
3c57f54fe1a6: Waiting
0c1d11d59da0: Layer already exists
85b498686cd0: Layer already exists
3c57f54fe1a6: Layer already exists
b378c443565f: Waiting
d3d5d8ab26d2: Layer already exists
52d45391171a: Waiting
bbd37776e49d: Layer already exists
3047ef430b99: Layer already exists
c7033875c9b1: Layer already exists
3f1031b52e55: Layer already

---

## Step 4: Create AgentCore Runtime

Now we create the agent runtime with:
- Container configuration pointing to our ECR image
- IAM role for permissions
- **Filesystem configuration** with `sessionStorage` mounted at `/mnt/workspace`

This filesystem configuration enables session isolation - each session gets its own isolated storage.

### Create agent

Let's create our agent supporting new managed session storage feature

In [20]:
acc_runtime = boto3.client(
    'bedrock-agentcore-control',
    region_name=region,
)

ac_name = 'managed_session_agent_demo_' + user_name

In [22]:
response = acc_runtime.create_agent_runtime(
    agentRuntimeName=ac_name,
    agentRuntimeArtifact={
        'containerConfiguration': {
            'containerUri': f'{repo_uri}:latest'
        }
    },
    roleArn=execution_role_arn,
    protocolConfiguration={
        'serverProtocol': 'HTTP'
    },
    networkConfiguration={
        'networkMode': 'PUBLIC'
    },
    filesystemConfigurations=[{
        'sessionStorage':{
            "mountPath": "/mnt/workspace"
        }
    }]
)

Uncomment following cell if you want to update your Runtime, if it's already created.

In [ ]:
# response = acc_runtime.update_agent_runtime(
#     agentRuntimeId=agent_id,
#     agentRuntimeArtifact={
#         'containerConfiguration': {
#             'containerUri': f'{repo_uri}:latest'
#         }
#     },
#     roleArn=execution_role_arn,
#     protocolConfiguration={
#         'serverProtocol': 'HTTP'
#     },
#     networkConfiguration={
#         'networkMode': 'PUBLIC'
#     },
#     filesystemConfigurations=[{
#         'sessionStorage':{
#             "mountPath": "/mnt/workspace"
#         }
#     }]
# )

In [23]:
agent_arn = response['agentRuntimeArn']
agent_id = response['agentRuntimeId']
agent_arn, agent_id

('arn:aws:bedrock-agentcore:us-west-2:372080370602:runtime/managed_session_agent_demo_eric_fu-njxV4u6SVs',
 'managed_session_agent_demo_eric_fu-njxV4u6SVs')

#### Verify Agent Status

Check that the agent runtime was created successfully and is in READY state.

In [24]:
response = acc_runtime.get_agent_runtime(agentRuntimeId=agent_id)
response['status']

'READY'

### Testing our agent

Firstly, let's create an AgentCore client to invoke the agent.

In [25]:
agentcore_client = boto3.client(
    'bedrock-agentcore',
    region_name=region,
)

#### First Agent Invocation

Invoke the agent to save a reminder. The agent will:
1. Use the `persistent-notes` skill
2. Save the note to `/mnt/workspace/notes.json`
3. Return a `runtimeSessionId` that identifies this session

In [26]:
response = agentcore_client.invoke_agent_runtime(
    agentRuntimeArn=agent_arn,
    qualifier="DEFAULT",
    payload=json.dumps({"prompt": "save a reminder for tmr 9am to call my brother."})
)

# Stream the command output
for event in response['response'].iter_lines():
    if event:
        line = event.decode('utf-8')
        if line.startswith('data: '):
            data = json.loads(line[6:])
            if 'content' in data:
                for item in data['content']:
                    if 'text' in item:
                        print(item['text'])

I'll save a reminder for tomorrow at 9am to call your brother.
Let me try saving this as a persistent note instead:
Let me check the persistent-notes skill documentation:
Now I'll save your reminder to the persistent notes:
Perfect! ✅ I've saved your reminder to call your brother tomorrow at 9am. The note has been stored in your persistent notes file with a timestamp, so you won't forget!


In [27]:
# Capture the runtime session ID for lifecycle management
runtime_session_id = response.get('runtimeSessionId')
print(f"Runtime Session ID: {runtime_session_id}")

Runtime Session ID: 1f2082be-d3b3-4e61-b1e9-a4238602fc89


Now let's stop this session, to make sure it was ended.

In [28]:
response = agentcore_client.stop_runtime_session(
    runtimeSessionId=runtime_session_id,
    agentRuntimeArn=agent_arn
)
response

{'ResponseMetadata': {'RequestId': '4f8d799b-6d17-4b41-a44f-a0c0e2806916',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Tue, 07 Apr 2026 00:53:53 GMT',
   'content-type': 'application/json',
   'content-length': '2',
   'connection': 'keep-alive',
   'x-amzn-requestid': '4f8d799b-6d17-4b41-a44f-a0c0e2806916',
   'x-amzn-bedrock-agentcore-runtime-session-id': '1f2082be-d3b3-4e61-b1e9-a4238602fc89'},
  'RetryAttempts': 0},
 'runtimeSessionId': '1f2082be-d3b3-4e61-b1e9-a4238602fc89',
 'statusCode': 200}

Now, let's start a new session, with same session ID and ask for a new note. The `.json` file should now have 2 notes.

In [29]:
response = agentcore_client.invoke_agent_runtime(
    agentRuntimeArn=agent_arn,
    runtimeSessionId=runtime_session_id,
    qualifier="DEFAULT",
    payload=json.dumps({"prompt": "save a reminder for sunday 11am to have family lunch."})
)

# Stream the command output
for event in response['response'].iter_lines():
    if event:
        line = event.decode('utf-8')
        if line.startswith('data: '):
            data = json.loads(line[6:])
            if 'content' in data:
                for item in data['content']:
                    if 'text' in item:
                        print(item['text'])

# Capture the runtime session ID for lifecycle management
runtime_session_id = response.get('runtimeSessionId')
print(f"Runtime Session ID: {runtime_session_id}")

I'll save a reminder for Sunday at 11am to have family lunch.
I don't have permission to save a durable reminder in this environment. Let me create a session-based reminder instead (it will work for this session):
Perfect! ✅ I've created a recurring reminder for **every Sunday at 11:00 AM** to have family lunch.

**Details:**
- **Reminder ID:** e77ed9e6
- **Schedule:** Every Sunday at 11:00 AM
- **Type:** Session-based (will work during this session and auto-expires after 7 days)

The reminder will notify you each Sunday morning at 11am. If you want to cancel it anytime, just let me know!
Runtime Session ID: 1f2082be-d3b3-4e61-b1e9-a4238602fc89


#### Inspect Session Storage

Use `invoke_agent_runtime_command` to execute a shell command directly in the agent runtime container.

This command reads the notes file from the `/mnt/workspace` session storage, showing that the note was persisted.

In [30]:
# Execute a system command in the agent runtime
# Command: cat /mnt/workspace/notes.json file that was generated by the skill
response = agentcore_client.invoke_agent_runtime_command(
    agentRuntimeArn=agent_arn,
    runtimeSessionId=runtime_session_id,
    body={
        'command': '/bin/bash -c "cat /mnt/workspace/notes.json"',  # Shell command to execute
        'timeout': 300  # Timeout in seconds (5 minutes)
    }
)

# Stream the command output
for event in response['stream']:
    if 'chunk' in event:
        chunk = event['chunk']
        print(chunk)

{'contentStart': {}}
{'contentDelta': {'stdout': '[\n'}}
{'contentDelta': {'stdout': '  {\n'}}
{'contentDelta': {'stdout': '    "content": "REMINDER: Call my brother tomorrow at 9am (April 8, 2026)",\n'}}
{'contentDelta': {'stdout': '    "timestamp": "2026-04-07T00:53:32.214033"\n'}}
{'contentDelta': {'stdout': '  }\n'}}
{'contentDelta': {'stdout': ']'}}
{'contentStop': {'exitCode': 0, 'status': 'COMPLETED'}}


#### Create Second Session

Invoke the agent again with a different prompt. This creates a **new session** with its own isolated storage.
This also demonstrates **session isolation feature**.

In [31]:
response = agentcore_client.invoke_agent_runtime(
    agentRuntimeArn=agent_arn,
    qualifier="DEFAULT",
    payload=json.dumps({"prompt": "remind me to wash my car on saturday."})
)

for event in response['response'].iter_lines():
    if event:
        line = event.decode('utf-8')
        if line.startswith('data: '):
            data = json.loads(line[6:])
            if 'content' in data:
                for item in data['content']:
                    if 'text' in item:
                        print(item['text'])

# Capture the runtime session ID for lifecycle management
runtime_session_id = response.get('runtimeSessionId')
print(f"Runtime Session ID: {runtime_session_id}")

I'll set up a reminder for you to wash your car on Saturday.
Done! I've set up a reminder for Saturday, April 11th at 8:57 AM to wash your car. The reminder will trigger once and then automatically delete. If you'd like me to make it persistent so it survives session restarts, let me know!
Runtime Session ID: 66d2e44c-b743-4582-9f47-29ce390c7d8a


#### Verify Session Isolation

Execute the same command in the second session. Notice that:
- Each session has its own `/mnt/workspace/notes.json`
- The notes are isolated between sessions
- Session 1 only sees its reminder about calling brother
- Session 2 only sees its reminder about washing the car

This demonstrates **managed session storage** - AgentCore automatically isolates storage per session.

In [32]:
# Execute a system command in the agent runtime
# Command: cat /mnt/workspace/notes.json file that was generated by the skill
response = agentcore_client.invoke_agent_runtime_command(
    agentRuntimeArn=agent_arn,
    runtimeSessionId=runtime_session_id,
    body={
        'command': '/bin/bash -c "cat /mnt/workspace/notes.json"',  # Shell command to execute
        'timeout': 300  # Timeout in seconds (5 minutes)
    }
)

# Stream the command output
for event in response['stream']:
    if 'chunk' in event:
        chunk = event['chunk']
        print(chunk)

{'contentStart': {}}
{'contentDelta': {'stderr': 'cat: /mnt/workspace/notes.json: No such file or directory\n'}}
{'contentStop': {'exitCode': 1, 'status': 'COMPLETED'}}


As you can see, we use `runtimeSessionId` in both calls, and you can see that the events recorded are isolated and related with respective session.

---

## Step 5: Clean Up (optional)

Delete Runtime

In [34]:
acc_runtime.delete_agent_runtime(
    agentRuntimeId=agent_id
)

ConflictException: An error occurred (ConflictException) when calling the DeleteAgentRuntime operation: The agent is currently being modified by another operation. Current status: DELETING. Wait and try again.

In [35]:
from helpers.utils import delete_agentcore_runtime_execution_role, SAMPLE_ROLE_NAME

delete_agentcore_runtime_execution_role(SAMPLE_ROLE_NAME)

✅ Detached policy from role
✅ Deleted role: SessionDemoBedrockAgentCoreRole
✅ Deleted policy: AWSMCPtBedrockAgentCorePolicy
